# fetch alerts with Babamul api and crossmatch / filter

Adaped directly from: https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/notebook.ipynb

In [ ]:
# TODOS

# Make run daily, logging for which dates have been run, number filtered etc.
# create group on fritz and save objects there
# fully automate up to sending objects to fritz
# filter cosmos catalog based on recs here: https://cosmos2025.iap.fr/tutorials/catalog_usage_tutorial.html
# get other catalogs relevant for crossmatches with other fields (EDFS, M49)
# make filter version w/ fewer detections for anirudh (jet, grb afterglow))
# second pass to improve TDE filter, check other objects?
# redownload cosmos2025 and save more catalog values (ie z high and low, chi2, etc)

In [ ]:
#imports 
from astropy.table import Table
from datetime import datetime
import babamul
from babamul import jupyter
import dotenv
dotenv.load_dotenv()

In [ ]:
import importlib
import rubin_stats_functions
import filter_functions
importlib.reload(rubin_stats_functions)
importlib.reload(filter_functions)

### Catalog uniformization

cut to subsection of confident galaxies

rename columns with universal schema (id, ra, dec, z, + any custom values that are useful)

In [ ]:
from astropy.table import Table                                                                                                                                             
from astropy.io import fits
import numpy as np

In [ ]:
with fits.open('data/catalogs/COSMOS_11bands-SExtractor-Lephare.fits') as hdul:
    hdul.info()


In [ ]:
t = Table.read('data/catalogs/XMMLSS_11bands-SExtractor-Lephare.fits')                                                                                                 
t[0:5]


In [ ]:
# histogram t table ZPHOT values

import matplotlib.pyplot as plt

plt.hist(t['ZPHOT'], bins=50, range=(0, 6))

In [ ]:
galaxies = t[
    (t['OBJ_TYPE'] == 0) &
    (t['CHI_BEST'] > 0) &        # has a valid SED fit
    (t['Z_BEST'] > 0) &          # has a valid photo-z
    (t['MASK'] == 0) &           # not in a bad region
    (t['CLASS_STAR_HSC_I'] < 0.9)  # morphologically not a star
]

print(f'total objects: {len(t)}')
print(len(galaxies))

In [ ]:
cut_catalog = galaxies['ID', 'RA', 'DEC', 'ZPHOT',
                        'Z_BEST68_LOW', 'Z_BEST68_HIGH', 'CHI_BEST',
                        'OBJ_TYPE', 'CLASS_STAR_HSC_I',
                        ].copy()

cut_catalog.rename_column('ID', 'id')
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog.rename_column('ZPHOT', 'z')

print(len(cut_catalog))

In [ ]:
for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

In [ ]:
cut_catalog.write('data/catalogs/CLAUDS_SourceExtractor_XMMLSS_cut.fits', overwrite=False) 

### catalog crossmatch

Develop pipeline to crossmatch (or do any inference) on sources outside of Fritz, and save back to Fritz

In [ ]:
# get alerts from date range and other basic filters for astrophysical sources
# this has crashed for large date ranges, so is built to save in batches.

from rubin_stats_functions import babamul_get_alerts

start="06-17-2026"
end="06-29-2026"

loaded_alerts = babamul_get_alerts(
    survey="LSST",
    start_time=start,
    end_time=end,
    min_drb=0.4,
    is_rock=False,
    is_star=False,
    is_near_brightstar=False,
    is_stationary=True,
) 

In [ ]:
#if we saved batches as we fetched, combine the batched files of alerts into a single compressed file
from rubin_stats_functions import combine_alert_files

start_fmt = datetime.strptime(start, "%m-%d-%Y").strftime("%m%d%y")                                                                                                         
end_fmt = datetime.strptime(end, "%m-%d-%Y").strftime("%m%d%y")
filename = f"basic_{start_fmt}_{end_fmt}.json.gz" 

combined = combine_alert_files(input_dir="data/lsst_alert_download/raw_files/", output_path=f"data/lsst_alert_download/{filename}", delete_raw=True)

In [ ]:
# bookkeeping - combine saved alerts to bigger batches per file
from rubin_stats_functions import combine_alert_files

combined = combine_alert_files(                                                                                                                                           
    input_dir="data/lsst_alert_download/",                                                                                                                      
    input_files=["basic_041226_061726.json.gz", "basic_061726_062926.json.gz"],
    output_path="data/lsst_alert_download/basic_041226_062926.json.gz",
    delete_raw=True                                                                                                         
)   

In [ ]:
# if we want to open saved alerts file, we can do that too

from rubin_stats_functions import load_alerts
loaded_alerts = load_alerts("data/lsst_alert_download/basic_041226_062926.json.gz")

In [ ]:
# additional filtering

# in generic_filter we make some more generic cuts to get real astrophysical objects
# filter out dipole (image subtraction artifacts), poor PSF fits, low SNR, extended sources, and alerts with shape or centroid fitting issues. 
# filter currently repeats cuts made in babamul_get_alerts: drb<0.4, rock, star, near_brightstar, isdiffpos, and stationary.

filtered_alerts = filter_functions.filter_alerts(                                                                                                           
    loaded_alerts,                                                            
    filter_functions.generic_filter,                                                                                                                        
) 

# deduplicate alerts to get unique objects
filtered_objects = filter_functions.deduplicate_alerts(filtered_alerts)

### Catalog crossmatch

In [ ]:
import importlib
import rubin_stats_functions
import filter_functions
importlib.reload(rubin_stats_functions)
importlib.reload(filter_functions)

In [ ]:
# crossmatch with all available catalogs. returns a dictionary with all matched alert and catalog info.

crossmatched_objects = filter_functions.catalog_crossmatch(
      alerts=filtered_objects,
      catalog_name=[
          "CLAUDS_SourceExtractor_XMMLSS_cut.fits",
          "CLAUDS_SourceExtractor_DEEP23_cut.fits"
      ],
)

In [ ]:
# additional filtering on catalog features

from filter_functions import catalog_filter

filtered = catalog_filter(
    crossmatched_objects,
    z_min = 0.5,
)

In [ ]:
# Source specific filtering:
# in tde_filter we also make more specific cuts, including minimum x detections (specfic to DDFs), no milliquas matches

filtered = filter_functions.filter_alerts(                                                                                                           
    filtered,                                                            
    filter_functions.tde_filter,
) 

### loading and inspecting objects

In [ ]:
crossmatched_alerts = [v["LSST"] for v in crossmatched_objects.values()]
print(f"Total alerts: {len(crossmatched_alerts)}")

filtered_crossmatched_alerts = [v["LSST"] for v in filtered.values()]  
print(f"Filtered alerts: {len(filtered_crossmatched_alerts)}")

In [ ]:
# inspect all crossmatch results without the cuts on rise etc.

from astropy.time import Time
print(f'Todays JD: {round(Time.now().jd)}')
babamul.jupyter.scan_alerts(filtered_crossmatched_alerts)

In [ ]:
# print crossmatch info for a given id 

id = '170591539645907039'

obj = crossmatched_objects[id]
for catalog, data in obj.items():
     if catalog != "LSST" and data is not None:
         print(f"{catalog}:")
         print(f"end=z = {data['z']:.2f}") 
         print(f"chi2 = {data['CHI_BEST']:.2f}")
         print(f"separation = {data['consearch_arcsecs']:.2f} arcsec")


In [ ]:
# by hand create a list of objects that pass inspection, save
# will add objectid to existing file, without duplicating objects

cand_ids_to_save = []

from rubin_stats_functions import save_objects                                                                                                                     
save_objects(cand_ids_to_save, path="data/tde_candidates.json.gz")

### additional ways to fetch objects

In [ ]:
# or we provide function to run cosmos crossmatch by directly provided ra and dec instead of alert
# you may provide a single ra and dec, or a list of ras and a list of corresponding decs as arguments.

crossmatched_coords = filter_functions.catalog_crossmatch(
    ra=150.101273,
    dec=2.743894,
    radius_arcsec=300,
    catalog_path="data/catalogs",
)

In [ ]:
# or we can load our list of previously saved candidates, and fetch alerts to display

from rubin_stats_functions import load_objects, fetch_latest_alerts                                                                                                

object_ids = load_objects("data/tde_candidates.json.gz")                                                                                                           
latest_alerts = fetch_latest_alerts(object_ids)     
saved_objects = filter_functions.catalog_crossmatch(latest_alerts)   

### prost

https://github.com/alexandergagliano/Prost

notes: set priors on z, offset, host mag.

missed catalog probability: true host too faint and undetected in image

small cone probability: search radius too small

can return top 2 best host associations.

In [ ]:
# FIXME: make normal conesearch on relatively large radius (eg 15"), return all matches
# for matches, run prost to select match w/ probability

In [ ]:
import pandas as pd
from astro_prost import associate_sample
from scipy.stats import gamma, halfnorm, uniform

# define a transient catalog 
transient_catalog = pd.DataFrame({
    'name': ['MyTransient'],
    'ra': [350.838161],
    'dec': [0.101394]
})

# define a set of catalogs to search -- options are glade, decals, panstarrs, and skymapper
catalogs = ["decals"]

# define priors and likelihoods
priorfunc_offset = uniform(loc=0, scale=10)
likefunc_offset = gamma(a=0.75)

priors = {"offset": priorfunc_offset}
likes = {"offset": likefunc_offset}

# associate
hosts = \
    associate_sample(
        transient_catalog,
        priors=priors,
        likes=likes,
        catalogs=catalogs,
        name_col='name',
        coord_cols=('ra', 'dec'),
        save=False
)